# Phase 6 - Dual-Path Stage 1 Evaluation: 3-MODE Apples-to-Apples Comparison

Run multiple Stage 1 variants under the EXACT SAME evaluation conditions
(same val data, same Stage 2 diffusion decoder, same metrics) so we can
compare model contributions fairly.

Modes (controlled by `MODES_TO_RUN` in Cell 2):

| mode                | Stage 1 stack                                  | sigma_data |
|---------------------|------------------------------------------------|------------|
| `phase5_oracle`     | encoder + RCN + reg_head (Phase 5 SRE ckpt)    | 0.18856    |
| `phase6_pathA_only` | encoder + RCN + reg_head (Phase 6 ckpt, mu_A)  | 0.18856    |
| `phase6_dualpath`   | encoder + RCN + reg_head + dual_path (mu_total)| 0.193      |
| `noncausal`         | RegressionMeanPredictor (CorrDiff baseline)    | from ckpt  |

Inputs
------
- `epoch_best_stage1_with_sre_best.pth` : Phase 5 best (encoder+RCN+head+SRE)
- `epoch_best_dualpath.pth`             : Phase 6 (encoder+RCN+head+dual_path)
- `epoch_last.pth`                      : Stage 2 diffusion weights (shared)
- (optional) `ckpt_noncausal/epoch_last.pth` : CorrDiff baseline

Output
------
- `phase6_evaluation/phase6_eval_3modes_apples.json`

Compute estimate
----------------
- N_TIMES_EVAL=365 x K_SAMPLES_EVAL=12 x N_STEPS_DIFF=18 = ~78k diffusion calls per mode
- T4 fp32 ~90 min per mode -> ~4.5h for 3 modes, ~6h for 4 modes


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab - Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Imports + Constants + Modes List ===
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from pathlib import Path
from omegaconf import OmegaConf

from st_cdgm.models.dual_path_stage1 import DualPathPredictor, PathBCNN, FusionGate
from st_cdgm.training.stage1_paths import (
    predict_mu_hr_dualpath,
    batch_lr_grid_last,
    _as_batched_hr,
)

# --- Drive paths ---
DRIVE_ROOT     = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N      = DRIVE_ROOT / 'oracle_9node' / 'seed_42'

# --- Stage 1 checkpoints (one per mode) ---
CKPT_PHASE5    = ORACLE_9N / 'epoch_best_stage1_with_sre_best.pth'
CKPT_DUALPATH  = ORACLE_9N / 'epoch_best_dualpath.pth'

# --- Stage 2 (diffusion) checkpoint - SHARED across all 3 Stage 1 variants ---
CKPT_STAGE2    = ORACLE_9N / 'epoch_last.pth'

# --- Non-causal CorrDiff baseline (optional). Probe both layouts. ---
CKPT_NONCAUSAL_CANDIDATES = [
    ORACLE_9N / 'ckpt_noncausal' / 'epoch_last.pth',
    DRIVE_ROOT / 'oracle_full' / 'seed_42' / 'ckpt_noncausal' / 'epoch_last.pth',
]
CKPT_NONCAUSAL = next((p for p in CKPT_NONCAUSAL_CANDIDATES if p.exists()), None)

# --- sigma_data per mode ---
SIGMA_DATA_PHASE5      = 0.18856  # Phase 5 original calibration (mu_A only)
SIGMA_DATA_DUALPATH    = 0.193    # Phase 6 recalibration (mu_total)
# noncausal uses sigma_data from its own checkpoint - no override

SIGMA_DATA_BY_MODE = {
    'phase5_oracle':     SIGMA_DATA_PHASE5,
    'phase6_pathA_only': SIGMA_DATA_PHASE5,   # Path A only -> same regime as Phase 5
    'phase6_dualpath':   SIGMA_DATA_DUALPATH,
    'noncausal':         None,                # signal: keep ckpt default
}

# --- Modes to run, in this order ---
MODES_TO_RUN = ['phase5_oracle', 'phase6_pathA_only', 'phase6_dualpath']
# Append 'noncausal' only if checkpoint exists (will be skipped gracefully otherwise).
if CKPT_NONCAUSAL is not None:
    MODES_TO_RUN.append('noncausal')

# --- Eval params (match v5_eval Cell 10 protocol) ---
N_TIMES_EVAL    = 365      # full year of val data
K_SAMPLES_EVAL  = 12       # ensemble size
N_STEPS_DIFF    = 18       # diffusion sampling steps

# --- Output dir ---
OUT_DIR        = ORACLE_9N / 'phase6_evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Validations (Phase 5 + Phase 6 + Stage 2 are required; noncausal optional) ---
_missing_ckpts = []
if not CKPT_PHASE5.exists():    _missing_ckpts.append(('CKPT_PHASE5',   CKPT_PHASE5))
if not CKPT_DUALPATH.exists():  _missing_ckpts.append(('CKPT_DUALPATH', CKPT_DUALPATH))
if not CKPT_STAGE2.exists():    _missing_ckpts.append(('CKPT_STAGE2',   CKPT_STAGE2))
if _missing_ckpts:
    # Compute which modes can still run.
    _can_run = []
    if CKPT_PHASE5.exists() and CKPT_STAGE2.exists():
        _can_run.append('phase5_oracle')
    if CKPT_DUALPATH.exists() and CKPT_STAGE2.exists():
        _can_run += ['phase6_pathA_only', 'phase6_dualpath']
    print('[Cell 2] WARNING - missing checkpoints:')
    for name, path in _missing_ckpts:
        print(f'  {name} = {path}')
    print(f'[Cell 2] Modes that can run with the available files: {_can_run}')
    MODES_TO_RUN = [m for m in MODES_TO_RUN if m in _can_run or m == 'noncausal']

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cell 2] DEVICE         = {DEVICE}')
print(f'[Cell 2] CKPT_PHASE5    = {CKPT_PHASE5}  exists={CKPT_PHASE5.exists()}')
print(f'[Cell 2] CKPT_DUALPATH  = {CKPT_DUALPATH}  exists={CKPT_DUALPATH.exists()}')
print(f'[Cell 2] CKPT_STAGE2    = {CKPT_STAGE2}  exists={CKPT_STAGE2.exists()}')
print(f'[Cell 2] CKPT_NONCAUSAL = {CKPT_NONCAUSAL}')
print(f'[Cell 2] MODES_TO_RUN   = {MODES_TO_RUN}')
print(f'[Cell 2] N_TIMES_EVAL   = {N_TIMES_EVAL}  K = {K_SAMPLES_EVAL}  steps_diff = {N_STEPS_DIFF}')
print(f'[Cell 2] Total diffusion calls per mode: {N_TIMES_EVAL * K_SAMPLES_EVAL * N_STEPS_DIFF:,}')
print(f'[Cell 2] OUT_DIR        = {OUT_DIR}')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Forcer batch_size=1 pour la compatibilite single-sample (IterableDataset)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# --- Ajout metapaths 9-node ---
OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- Dates ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

# --- convert_sample_to_batch (identique training, avec lr_grid pour batch_lr_grid_last) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

# Constantes pour HR shape utilises plus tard pour DualPathPredictor
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Build shared Stage 2 (diffusion decoder), helpers for per-mode sigma_data ===
# Stage 2 weights are loaded ONCE from CKPT_STAGE2 and reused across all modes.
# Each mode overrides diffusion_decoder.edm_config.sigma_data right before its
# inference loop runs (via set_sigma_data_for_mode below).
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

print(f'[Cell 4] Loading Stage 2 : {CKPT_STAGE2}')
ck_s2 = torch.load(CKPT_STAGE2, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] keys[:12] = {sorted(ck_s2.keys())[:12]}')

# --- Detect num_vars from Phase 6 dualpath ckpt encoder metapaths ---
# We use this for the projection_class_embeddings_input_dim override the
# 9-node Stage 2 training expects. The Phase 5 and Phase 6 encoders share
# the same metapath set (same 9-node graph), so this number is reused.
_ck_peek = torch.load(CKPT_DUALPATH, map_location='cpu', weights_only=False)
_enc_sd_peek = _ck_peek.get('encoder_state_dict', {})
_metapath_names = set()
for k in _enc_sd_peek:
    if k.startswith('metapath_convs.'):
        _metapath_names.add(k[len('metapath_convs.'):].split('__')[0])
num_vars = len(_metapath_names)
del _ck_peek, _enc_sd_peek
print(f'[Cell 4] num_vars detected = {num_vars}')

# --- Build UNET_KWARGS with projection_class_embeddings_input_dim override ---
UNET_KWARGS = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KWARGS and isinstance(UNET_KWARGS[_k], list):
        UNET_KWARGS[_k] = tuple(UNET_KWARGS[_k])
UNET_KWARGS['projection_class_embeddings_input_dim'] = num_vars * int(CONFIG.diffusion.conditioning_dim)
print(f'[Cell 4] projection_class_embeddings_input_dim = {UNET_KWARGS["projection_class_embeddings_input_dim"]}')

# --- Build diffusion module ---
edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])

diffusion_decoder = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height),
    width=int(CONFIG.diffusion.width),
    unet_kwargs=UNET_KWARGS,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=bool(CONFIG.diffusion.get('use_gradient_checkpointing', False)),
    conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
    edm_config=edm_cfg,
    causal_concat=True,
).to(DEVICE)

# --- Strip torch.compile/DDP prefixes from state_dicts ---
def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out

_diff_sd = _strip_prefixes(ck_s2.get('diffusion_state_dict'))
if _diff_sd is None:
    raise RuntimeError(
        f'diffusion_state_dict absent from {CKPT_STAGE2}. Stage 2 ckpt must be '
        f'the epoch_last.pth from Path C+ Option C 9-node training.'
    )
_missing, _unexpected = diffusion_decoder.load_state_dict(_diff_sd, strict=False)
if _missing:
    print(f'[Cell 4] missing keys : {len(_missing)} (first 3: {_missing[:3]})')
if _unexpected:
    print(f'[Cell 4] unexpected keys : {len(_unexpected)} (first 3: {_unexpected[:3]})')

# Save original (calibrated by stage-2 training) sigma_data so we can restore for noncausal.
_SIGMA_DATA_CKPT = float(diffusion_decoder.edm_config.sigma_data)
print(f'[Cell 4] sigma_data from Stage 2 ckpt = {_SIGMA_DATA_CKPT:.5f}')

for p in diffusion_decoder.parameters():
    p.requires_grad_(False)
diffusion_decoder.eval()

_n_diff = sum(p.numel() for p in diffusion_decoder.parameters())
print(f'[Cell 4] Stage 2 loaded, diffusion params = {_n_diff:,}')


def set_sigma_data_for_mode(mode_name):
    """Override diffusion_decoder.edm_config.sigma_data per mode mapping.

    Returns the value actually used (so it can be logged + stored).
    For 'noncausal' (or any mode mapped to None), restores the ckpt value.
    """
    new = SIGMA_DATA_BY_MODE.get(mode_name)
    if new is None:
        # Restore ckpt-original value
        diffusion_decoder.edm_config.sigma_data = _SIGMA_DATA_CKPT
        print(f'[set_sigma_data_for_mode] mode={mode_name}: sigma_data restored to ckpt value {_SIGMA_DATA_CKPT:.5f}')
        return _SIGMA_DATA_CKPT
    diffusion_decoder.edm_config.sigma_data = float(new)
    print(f'[set_sigma_data_for_mode] mode={mode_name}: sigma_data = {float(new):.5f}')
    return float(new)


In [ ]:
# === Cell 5 : Stage 1 builders (one per mode), with safe checkpoint loading ===
# Each builder returns a tuple (encoder, rcn_cell, rcn_runner, regression_head, dual_path_or_None)
# that the run loop in Cell 6 consumes. The Phase 5 / phase6_pathA_only / phase6_dualpath
# stacks share the same encoder + RCN + head pattern (auto-detected from the ckpt);
# only the dual_path module is loaded for phase6_dualpath. The noncausal mode builds
# a RegressionMeanPredictor instead.
import re
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder


def _parse_encoder_metapaths_from_ckpt(enc_sd):
    """Infer ordered metapath list from encoder state_dict keys.

    Key format: ``metapath_convs.{name}__{src}__{rel}__{target}.{param}``.
    """
    seen = {}
    order = []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]
        src  = parts[1]
        rel  = parts[2]
        tgt  = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt)
            order.append(name)
    return [(n,) + seen[n] for n in order]


def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd


def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] loaded from "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] FAILED with "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] no valid key found in {keys}')
    return False


def _build_encoder_from_ckpt(enc_sd, device):
    parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
    print(f'  metapaths detected : {[t[0] for t in parsed]}')
    cfgs = [
        IntelligibleVariableConfig(name=n, meta_path=(s, r, t), pool='mean')
        for n, s, r, t in parsed
    ]
    enc = IntelligibleVariableEncoder(
        configs=cfgs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(device)
    return enc, len(cfgs)


def _build_causal_stack(ckpt_path, device, with_dual_path=False):
    """Build encoder + RCN + reg_head (and optionally dual_path) from a ckpt.

    Used by phase5_oracle, phase6_pathA_only, phase6_dualpath.
    """
    print(f'[build_causal_stack] ckpt = {ckpt_path}  with_dual_path={with_dual_path}')
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f'  ckpt keys[:12] = {sorted(ck.keys())[:12]}')

    enc_sd = _clean_sd(ck.get('encoder_state_dict', {}))
    torch.manual_seed(SEED); np.random.seed(SEED)
    encoder, n_vars = _build_encoder_from_ckpt(enc_sd, device)

    _probe_b = next(iter(val_dataset))
    _lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
    rcn_driver_dim = _lr_nodes.shape[-1]

    rcn_cell = RCNCell(
        num_vars=n_vars,
        hidden_dim=int(CONFIG.rcn.hidden_dim),
        driver_dim=rcn_driver_dim,
        reconstruction_dim=rcn_driver_dim,
        dropout=float(CONFIG.rcn.dropout),
    ).to(device)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

    rh_cfg = CONFIG.two_stage.regression_head
    regression_head = GraphToGridDecoder(
        d_model=int(rh_cfg.d_model),
        hr_h=H_HR, hr_w=W_HR,
        intermediate_h=int(rh_cfg.intermediate_h),
        intermediate_w=int(rh_cfg.intermediate_w),
        n_heads=int(rh_cfg.n_heads),
        refine_channels=int(rh_cfg.refine_channels),
        output_channels=1,
    ).to(device)

    dual_path = None
    if with_dual_path:
        PATH_B_KIND          = 'unet'
        PATH_B_UNET_CHANNELS = (32, 64, 128)
        PATH_B_UNET_LR_SHAPE = (23, 26)
        PATH_B_BASE_CH       = 48
        GATE_MAX_MEAN        = 0.40
        dual_path = DualPathPredictor(
            in_channels=C_LR,
            base_ch=PATH_B_BASE_CH,
            hr_h=H_HR, hr_w=W_HR,
            gate_max_mean=GATE_MAX_MEAN,
            path_b_kind=PATH_B_KIND,
            path_b_unet_channels=PATH_B_UNET_CHANNELS,
            path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
        ).to(device)

    # Load weights
    _safe_load(encoder,         ck, ['encoder_state_dict'],                            'encoder')
    _safe_load(rcn_cell,        ck, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
    _safe_load(regression_head, ck, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
    if with_dual_path:
        _safe_load(dual_path,   ck, ['dual_path_state_dict'],                          'dual_path')

    # Freeze + eval
    mods = [encoder, rcn_cell, regression_head]
    if dual_path is not None:
        mods.append(dual_path)
    for m in mods:
        for p in m.parameters():
            p.requires_grad_(False)
        m.eval()

    # A_dag check
    _rcn_core = rcn_cell
    if hasattr(_rcn_core, '_orig_mod'):
        _rcn_core = _rcn_core._orig_mod
    if hasattr(_rcn_core, 'A_dag'):
        _A_dag = _rcn_core.A_dag.detach()
        print(f'  A_dag shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
              f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')

    _n_enc = sum(p.numel() for p in encoder.parameters())
    _n_rcn = sum(p.numel() for p in rcn_cell.parameters())
    _n_rh  = sum(p.numel() for p in regression_head.parameters())
    _n_dp  = sum(p.numel() for p in dual_path.parameters()) if dual_path is not None else 0
    print(f'  param counts: enc={_n_enc:,}  rcn={_n_rcn:,}  head={_n_rh:,}  dual_path={_n_dp:,}')
    if dual_path is not None:
        print(f'  path_b_bias = {float(dual_path.path_b_bias.item()):+.5f}')
    return encoder, rcn_cell, rcn_runner, regression_head, dual_path


def _build_noncausal_stack(ckpt_path, device):
    """Build the RegressionMeanPredictor baseline (Stage 1 only, no encoder/RCN).

    Returns (None, None, None, regression_head, None) so the run_mode loop
    can detect 'noncausal' simply by checking encoder is None.
    """
    print(f'[build_noncausal_stack] ckpt = {ckpt_path}')
    from st_cdgm.models.regression_mean_predictor import (
        RegressionMeanPredictor, RegressionPredictorConfig,
    )
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f'  ckpt keys[:12] = {sorted(ck.keys())[:12]}')

    cfg = RegressionPredictorConfig(
        in_channels=C_LR,
        out_channels=1,
        lr_height=int(lr_shape[0]),
        lr_width=int(lr_shape[1]),
        hr_height=H_HR,
        hr_width=W_HR,
        block_out_channels=(64, 128, 192),
        layers_per_block=2,
        norm_num_groups=16,
        state_adapter_hidden_dim=int(CONFIG.rcn.hidden_dim),
    )
    rmp = RegressionMeanPredictor(cfg).to(device)
    _safe_load(rmp, ck, ['regression_head_state_dict', 'head_state_dict'], 'regression_head(noncausal)')
    for p in rmp.parameters():
        p.requires_grad_(False)
    rmp.eval()
    _n = sum(p.numel() for p in rmp.parameters())
    print(f'  RegressionMeanPredictor params = {_n:,}')
    return None, None, None, rmp, None


def build_stack_for_mode(mode_name):
    """Return (encoder, rcn_cell, rcn_runner, regression_head, dual_path_or_None).

    Returns None on any failure (so the run loop can skip and continue).
    """
    try:
        if mode_name == 'phase5_oracle':
            return _build_causal_stack(CKPT_PHASE5, DEVICE, with_dual_path=False)
        if mode_name == 'phase6_pathA_only':
            return _build_causal_stack(CKPT_DUALPATH, DEVICE, with_dual_path=False)
        if mode_name == 'phase6_dualpath':
            return _build_causal_stack(CKPT_DUALPATH, DEVICE, with_dual_path=True)
        if mode_name == 'noncausal':
            if CKPT_NONCAUSAL is None:
                print(f'[build_stack_for_mode] noncausal: no ckpt found, SKIPPING')
                return None
            return _build_noncausal_stack(CKPT_NONCAUSAL, DEVICE)
        print(f'[build_stack_for_mode] unknown mode: {mode_name}')
        return None
    except Exception as e:
        print(f'[build_stack_for_mode] FAILED for {mode_name}: {type(e).__name__}: {e}')
        return None


print('[Cell 5] builders defined: phase5_oracle | phase6_pathA_only | phase6_dualpath | noncausal')


In [ ]:
# === Cell 6 : run_mode(...) + outer loop over MODES_TO_RUN ===
# Iterates val_dataloader for N_TIMES_EVAL samples, draws K_SAMPLES_EVAL samples
# per timestep, accumulates (K, T, H, W) ensemble + (T, H, W) truth + times list.
# Returns these tensors per mode so Cell 7/8 can compute metrics.
import time as _time
import numpy as _np

print(f'[Cell 6] Apples-to-apples protocol :')
print(f'  N_TIMES_EVAL    = {N_TIMES_EVAL}')
print(f'  K_SAMPLES_EVAL  = {K_SAMPLES_EVAL}')
print(f'  N_STEPS_DIFF    = {N_STEPS_DIFF}')
print(f'  Total diff calls per mode = {N_TIMES_EVAL * K_SAMPLES_EVAL * N_STEPS_DIFF:,}')


@torch.no_grad()
def _predict_mu_for_mode(mode_name, batch, encoder, rcn_runner, regression_head, dual_path):
    """Compute mu_total for one batch under the requested mode."""
    if mode_name == 'noncausal':
        # RegressionMeanPredictor takes the LR grid directly.
        lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
        lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
        mu = regression_head(lr_safe)
        if mu.dim() == 3:
            mu = mu.unsqueeze(0)
        return mu

    # All causal modes (phase5_oracle, phase6_pathA_only, phase6_dualpath)
    # share the same encoder + RCN + head forward.
    lr_data = batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    H_T     = seq_out.states[-1]
    mu_A    = regression_head(H_T)
    if mu_A.dim() == 3:
        mu_A = mu_A.unsqueeze(0)

    if mode_name == 'phase6_dualpath' and dual_path is not None:
        lr_grid  = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
        lr_safe  = torch.nan_to_num(lr_grid, nan=0.0)
        mu_total, _mu_B, _gate = dual_path(lr_safe, mu_A)
        return mu_total

    # phase5_oracle and phase6_pathA_only: skip dual_path, return mu_A as mu_total.
    return mu_A


@torch.no_grad()
def _predict_ensemble(mode_name, batch, K, n_steps,
                      encoder, rcn_runner, regression_head, dual_path):
    """K-sample HR ensemble via the shared Stage 2 diffusion decoder."""
    mu_total = _predict_mu_for_mode(mode_name, batch, encoder, rcn_runner,
                                     regression_head, dual_path)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    samples = []
    for _k in range(K):
        out = diffusion_decoder.sample(
            conditioning=None,
            num_steps=n_steps,
            scheduler_type='edm_karras',
            apply_constraints=False,
            mu_HR=mu_total,
            baseline_log=bl,
        )
        r = out.residual if hasattr(out, 'residual') else out
        hr_pred = bl + mu_total + r
        samples.append(hr_pred)
    return torch.stack(samples, dim=0)  # [K, 1, 1, H, W]


def run_mode(mode_name, encoder, rcn_cell, rcn_runner, regression_head, dual_path,
             diffusion_decoder, sigma_data):
    """Iterate val_dataloader N_TIMES_EVAL times under given mode.

    Returns (ens_log1p_KT, truth_log1p_T, times_list).
    """
    # Set Stage 2 sigma_data for this mode (must happen BEFORE the sample loop).
    diffusion_decoder.edm_config.sigma_data = float(sigma_data)
    print(f'[run_mode/{mode_name}] sigma_data set to {float(sigma_data):.5f}')

    ens_log_list, truth_log_list, times_list = [], [], []
    t0 = _time.time()
    n_seen = 0
    for bi, batch_list in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
        if n_seen >= N_TIMES_EVAL:
            break
        for micro in batch_list:
            if n_seen >= N_TIMES_EVAL:
                break
            ens = _predict_ensemble(
                mode_name, micro, K=K_SAMPLES_EVAL, n_steps=N_STEPS_DIFF,
                encoder=encoder, rcn_runner=rcn_runner,
                regression_head=regression_head, dual_path=dual_path,
            )
            while ens.dim() > 3:
                ens = ens.squeeze(1)
            ens_arr = ens.cpu().numpy()  # (K, H, W)

            target_res = micro['residual'][-1].to(DEVICE)
            bl_t = micro['baseline'][-1].to(DEVICE)
            if bl_t.dim() == target_res.dim() - 1:
                bl_t = bl_t.unsqueeze(0)
            bl_t = torch.nan_to_num(bl_t, nan=0.0)
            truth = (bl_t + target_res).cpu()
            while truth.dim() > 2:
                truth = truth.squeeze(0)

            ens_log_list.append(ens_arr)
            truth_log_list.append(truth.numpy())

            tv = micro.get('time')
            if tv is not None:
                try:
                    if hasattr(tv, '__len__') and not isinstance(tv, str):
                        tv = tv[-1]
                except Exception:
                    pass
                times_list.append(tv)
            else:
                times_list.append(None)

            n_seen += 1

        if n_seen % 20 == 0 and n_seen > 0:
            elapsed = _time.time() - t0
            rate = n_seen / max(elapsed, 1e-6)
            eta  = (N_TIMES_EVAL - n_seen) / max(rate, 1e-6)
            print(f'  [{mode_name}] {n_seen:3d}/{N_TIMES_EVAL}  '
                  f'elapsed={elapsed:5.0f}s  eta={eta:5.0f}s  '
                  f'({rate*60:.2f}/min)')

    ens_KT  = _np.stack(ens_log_list,    axis=1)  # (K, T, H, W)
    truth_T = _np.stack(truth_log_list,  axis=0)  # (T, H, W)
    dt = _time.time() - t0
    print(f'[run_mode/{mode_name}] DONE in {dt:.0f}s ({dt/60:.1f} min)  '
          f'ens={ens_KT.shape}  truth={truth_T.shape}  '
          f'times={len(times_list)} (non-null={sum(1 for t in times_list if t is not None)})')
    return ens_KT, truth_T, times_list


# --- Outer loop : build, run, store per-mode arrays ---
all_predictions = {}   # mode -> dict with ens, truth, times
all_meta = {}          # mode -> dict with sigma_data, ckpt, wallclock, status

for _mode in MODES_TO_RUN:
    print()
    print('=' * 70)
    print(f'>>> MODE = {_mode}')
    print('=' * 70)
    _t_mode = _time.time()
    _stack = build_stack_for_mode(_mode)
    if _stack is None:
        all_meta[_mode] = {
            'status':      'SKIPPED - checkpoint not found or build failed',
            'sigma_data':  None,
            'ckpt':        None,
            'wallclock_s': 0.0,
        }
        print(f'[outer] {_mode}: SKIPPED.')
        continue
    enc_m, rcn_m, run_m, head_m, dp_m = _stack
    sigma_m = set_sigma_data_for_mode(_mode)
    ens_m, truth_m, times_m = run_mode(
        _mode, enc_m, rcn_m, run_m, head_m, dp_m,
        diffusion_decoder, sigma_m,
    )
    all_predictions[_mode] = {
        'ens_log1p_KT':  ens_m,
        'truth_log1p_T': truth_m,
        'times':         times_m,
    }
    _ckpt_used = {
        'phase5_oracle':     str(CKPT_PHASE5),
        'phase6_pathA_only': str(CKPT_DUALPATH),
        'phase6_dualpath':   str(CKPT_DUALPATH),
        'noncausal':         str(CKPT_NONCAUSAL) if CKPT_NONCAUSAL else None,
    }.get(_mode)
    all_meta[_mode] = {
        'status':      'OK',
        'sigma_data':  float(sigma_m),
        'ckpt':        _ckpt_used,
        'wallclock_s': _time.time() - _t_mode,
    }
    # Free Stage 1 modules to reduce GPU pressure before next mode.
    del enc_m, rcn_m, run_m, head_m, dp_m, _stack
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print('=' * 70)
print(f'[Cell 6] All modes done. Predictions collected for: {list(all_predictions.keys())}')
print(f'[Cell 6] Status per mode :')
for _m, _meta in all_meta.items():
    print(f'  {_m:20s} status={_meta["status"]:10s} sigma_data={_meta["sigma_data"]} '
          f'wallclock={_meta["wallclock_s"]:.0f}s')


In [ ]:
# === Cell 7 : probabilistic_metrics per mode ===
import numpy as np


def _crps_empirical_fast(samples, obs):
    K = samples.shape[0]
    term1 = np.nanmean(np.abs(samples - obs[None]), axis=0)
    s = np.sort(samples, axis=0)
    k_idx = np.arange(1, K + 1).reshape((K,) + (1,) * (s.ndim - 1)).astype(np.float64)
    weights = 2.0 * k_idx - K - 1.0
    term2 = np.sum(weights * s, axis=0) / (K * K)
    return term1 - term2


def _crps_clim_per_pixel(truth):
    T = truth.shape[0]
    t_sorted = np.sort(truth, axis=0)
    k_idx = np.arange(1, T + 1).reshape((T, 1, 1)).astype(np.float64)
    weights = 2.0 * k_idx - T - 1.0
    return np.sum(weights * t_sorted, axis=0) / (T * T)


def _rank_histogram(samples, truth):
    K = samples.shape[0]
    rank = (samples < truth[None]).sum(axis=0).astype(np.int64)
    hist, _ = np.histogram(rank.flatten(), bins=np.arange(K + 2) - 0.5)
    return hist.astype(int).tolist()


def probabilistic_metrics(ens_log1p, truth_log1p):
    """Compute full probabilistic metric battery in mm/day (from log1p inputs)."""
    ens   = np.expm1(np.clip(ens_log1p.astype(np.float64),   0.0, None))
    truth = np.expm1(np.clip(truth_log1p.astype(np.float64), 0.0, None))

    pred_mean = ens.mean(axis=0)
    err2 = (pred_mean - truth) ** 2
    rmse_global = float(np.sqrt(np.nanmean(err2)))
    rmse_map_t = np.sqrt(np.nanmean(err2, axis=0))

    ens_var = ens.var(axis=0)
    spread_global = float(np.sqrt(np.nanmean(ens_var)))
    spread_skill_ratio = float(spread_global / max(rmse_global, 1e-9))

    crps_model = _crps_empirical_fast(ens, truth)
    crps_model_global = float(np.nanmean(crps_model))
    crps_clim_map = _crps_clim_per_pixel(truth)
    crps_clim_global = float(np.nanmean(crps_clim_map))
    crps_ss = 1.0 - crps_model_global / max(crps_clim_global, 1e-9)

    hist = _rank_histogram(ens, truth)
    K = int(ens.shape[0])
    expected_per_bin = float(truth.size / (K + 1))
    chi2_uniform = float(sum((c - expected_per_bin) ** 2 / expected_per_bin for c in hist))

    return {
        'K_samples': K,
        'n_times': int(ens.shape[1]),
        'grid': [int(truth.shape[-2]), int(truth.shape[-1])],
        'rmse_global_mm': rmse_global,
        'rmse_map_mean_mm': float(np.nanmean(rmse_map_t)),
        'rmse_map_max_mm': float(np.nanmax(rmse_map_t)),
        'spread_global_mm': spread_global,
        'spread_skill_ratio': spread_skill_ratio,
        'crps_model_global_mm': crps_model_global,
        'crps_clim_global_mm': crps_clim_global,
        'crps_skill_score': float(crps_ss),
        'rank_histogram': hist,
        'rank_histogram_bins': list(range(len(hist))),
        'rank_histogram_chi2_vs_uniform': chi2_uniform,
    }


prob_data_by_mode = {}

for _mode, _pred in all_predictions.items():
    print()
    print('=' * 60)
    print(f'PROBABILISTIC METRICS  [mode = {_mode}]')
    print('=' * 60)
    _pd = probabilistic_metrics(_pred['ens_log1p_KT'], _pred['truth_log1p_T'])
    prob_data_by_mode[_mode] = _pd
    for k, v in _pd.items():
        if isinstance(v, float):
            print(f'  {k:32s} = {v:.5f}')
        elif isinstance(v, int):
            print(f'  {k:32s} = {v}')
        elif isinstance(v, list) and len(v) <= 20:
            print(f'  {k:32s} = {v}')

print()
print(f'[Cell 7] Computed prob metrics for: {list(prob_data_by_mode.keys())}')


In [ ]:
# === Cell 8 : run_aligned_eval (indices + PSD) per mode ===
import numpy as np
from st_cdgm.evaluation.aligned_eval import run_aligned_eval


def _coerce_times_to_dt64(tlist, n):
    valid = [t for t in tlist if t is not None]
    if len(valid) == n:
        try:
            return np.array([np.datetime64(t) for t in valid], dtype='datetime64[D]')
        except Exception as _e:
            print(f'[Cell 8] datetime64 coercion failed ({_e}); using synthetic range')
    print(f'[Cell 8] times incomplete ({len(valid)}/{n}); fallback synthetic daily 2010-01-01+')
    return np.array([np.datetime64('2010-01-01') + np.timedelta64(i, 'D')
                     for i in range(n)], dtype='datetime64[D]')


aligned_data_by_mode = {}

for _mode, _pred in all_predictions.items():
    print()
    print('=' * 60)
    print(f'ALIGNED METRICS  [mode = {_mode}]')
    print('=' * 60)

    _times_arr = _coerce_times_to_dt64(_pred['times'], _pred['truth_log1p_T'].shape[0])
    _pred_mean_log1p = _pred['ens_log1p_KT'].mean(axis=0)  # (T, H, W)
    _out_json = OUT_DIR / f'aligned_metrics_{_mode}.json'

    try:
        _aligned = run_aligned_eval(
            pred_fields=_pred_mean_log1p,
            truth_fields=_pred['truth_log1p_T'],
            times=_times_arr,
            out_path=str(_out_json),
            gcm='ACCESS-CM2',
            run_variant=f'phase6_apples_{_mode}',
            in_distribution=True,
            space='log1p',
            thresh=1.0,
            k_samples=K_SAMPLES_EVAL,
            psd_nx=int(H_HR),
            psd_ny=int(W_HR),
        )
        aligned_data_by_mode[_mode] = _aligned
        for k, v in _aligned.items():
            if isinstance(v, dict):
                print(f'  {k} :')
                for kk, vv in v.items():
                    if isinstance(vv, float):
                        print(f'    {kk:28s} = {vv:.5f}')
                    else:
                        print(f'    {kk:28s} = {vv}')
            elif isinstance(v, float):
                print(f'  {k:32s} = {v:.5f}')
            else:
                print(f'  {k:32s} = {v}')
        print(f'  [Cell 8] aligned JSON saved : {_out_json}')
    except Exception as _e:
        print(f'  [Cell 8] FAILED for {_mode}: {type(_e).__name__}: {_e}')
        aligned_data_by_mode[_mode] = {'error': f'{type(_e).__name__}: {_e}'}

print()
print(f'[Cell 8] Aligned metrics done for: {list(aligned_data_by_mode.keys())}')


In [ ]:
# === Cell 9 : Build comparison table + save consolidated JSON ===
import json
from pathlib import Path

# --- Aggregate metrics per mode ---
all_results = {}
for _mode in MODES_TO_RUN:
    _meta = all_meta.get(_mode, {})
    _entry = {
        'status':            _meta.get('status', 'UNKNOWN'),
        'sigma_data':        _meta.get('sigma_data'),
        'ckpt':              _meta.get('ckpt'),
        'wallclock_s':       _meta.get('wallclock_s'),
        'probabilistic_mm':  prob_data_by_mode.get(_mode),
        'aligned':           aligned_data_by_mode.get(_mode),
    }
    all_results[_mode] = _entry

# Explicit skipped-noncausal sentinel if not in MODES_TO_RUN.
if 'noncausal' not in all_results:
    all_results['noncausal'] = {
        'status':            'SKIPPED - checkpoint not found',
        'sigma_data':        None,
        'ckpt':              None,
        'wallclock_s':       0.0,
        'probabilistic_mm':  None,
        'aligned':           None,
    }

consolidated = {
    'phase':           'phase6_3modes_apples_to_apples',
    'protocol':        'cgan_rampal_vendored',
    'modes_run':       MODES_TO_RUN,
    'n_times':         int(N_TIMES_EVAL),
    'k_samples':       int(K_SAMPLES_EVAL),
    'n_steps_diff':    int(N_STEPS_DIFF),
    'gcm':             'ACCESS-CM2',
    'in_distribution': True,
    'ckpt_stage2':     str(CKPT_STAGE2),
    'sigma_data_by_mode': {k: (float(v) if v is not None else None)
                           for k, v in SIGMA_DATA_BY_MODE.items()},
    'results':         all_results,
}

out_path = OUT_DIR / 'phase6_eval_3modes_apples.json'
out_path.write_text(json.dumps(consolidated, indent=2, default=str), encoding='utf-8')
print(f'[Cell 9] Consolidated JSON saved : {out_path}')


# --- Comparison table (side-by-side) ---
def _gv(d, *keys, default=float('nan')):
    if d is None:
        return default
    for k in keys:
        if k in d:
            try:
                return float(d[k])
            except Exception:
                return default
    return default


def _fmt(x, w=14):
    try:
        return f'{float(x):>{w}.5f}'
    except Exception:
        return f'{str(x):>{w}s}'


MODES_FOR_TABLE = ['phase5_oracle', 'phase6_pathA_only', 'phase6_dualpath', 'noncausal']
MODES_IN_TABLE = [m for m in MODES_FOR_TABLE if m in all_results]

print()
print('=' * 100)
print('COMPARISON TABLE (apples-to-apples, same val data, same Stage 2)')
print('=' * 100)

# Header
_hdr = f'  {"Metric":<26s} '
for _m in MODES_IN_TABLE:
    _hdr += f'{_m:>18s} '
print(_hdr)
print('  ' + '-' * (26 + 19 * len(MODES_IN_TABLE)))


def _row(label, getter, w=18):
    line = f'  {label:<26s} '
    for _m in MODES_IN_TABLE:
        v = getter(_m)
        line += _fmt(v, w=w) + ' '
    print(line)


# Probabilistic
_row('rmse_global_mm',         lambda m: _gv(all_results[m].get('probabilistic_mm'), 'rmse_global_mm'))
_row('crps_model_global_mm',   lambda m: _gv(all_results[m].get('probabilistic_mm'), 'crps_model_global_mm'))
_row('crps_skill_score',       lambda m: _gv(all_results[m].get('probabilistic_mm'), 'crps_skill_score'))
_row('spread_skill_ratio',     lambda m: _gv(all_results[m].get('probabilistic_mm'), 'spread_skill_ratio'))


def _aligned_get(m, *keys):
    _a = all_results[m].get('aligned') or {}
    _idx = _a.get('indices', {}) or {}
    return _gv(_idx, *keys)


_row('cdd_bias',     lambda m: _aligned_get(m, 'cdd_bias', 'CDD_bias', 'cdd'))
_row('rx1day_bias',  lambda m: _aligned_get(m, 'rx1day_bias', 'Rx1day_bias', 'rx1day'))
_row('r10day_bias',  lambda m: _aligned_get(m, 'r10day_bias', 'R10day_bias', 'r10day'))
_row('r95p_bias',    lambda m: _aligned_get(m, 'r95p_bias', 'R95p_bias', 'r95p'))
_row('sdii_bias',    lambda m: _aligned_get(m, 'sdii_bias', 'SDII_bias', 'sdii'))
_row('psd_distance', lambda m: _gv(all_results[m].get('aligned'), 'psd_distance'))

print('  ' + '-' * (26 + 19 * len(MODES_IN_TABLE)))

# --- Deltas: phase6_dualpath vs phase5_oracle (headline) ---
print()
print('=' * 80)
print('DELTAS: phase6_dualpath  -  phase5_oracle  (headline gain)')
print('=' * 80)
if ('phase6_dualpath' in all_results and all_results['phase6_dualpath'].get('status') == 'OK'
        and 'phase5_oracle' in all_results and all_results['phase5_oracle'].get('status') == 'OK'):
    def _delta(metric_getter, label):
        v_dp = metric_getter('phase6_dualpath')
        v_p5 = metric_getter('phase5_oracle')
        try:
            d = float(v_dp) - float(v_p5)
            print(f'  delta {label:<28s} = {d:+.5f}   '
                  f'(p5={float(v_p5):.5f} -> dp={float(v_dp):.5f})')
        except Exception:
            print(f'  delta {label:<28s} = N/A')

    _delta(lambda m: _gv(all_results[m].get('probabilistic_mm'), 'rmse_global_mm'),       'rmse_global_mm')
    _delta(lambda m: _gv(all_results[m].get('probabilistic_mm'), 'crps_model_global_mm'), 'crps_model_global_mm')
    _delta(lambda m: _gv(all_results[m].get('probabilistic_mm'), 'spread_skill_ratio'),   'spread_skill_ratio')
    _delta(lambda m: _aligned_get(m, 'cdd_bias', 'CDD_bias', 'cdd'),                       'cdd_bias')
    _delta(lambda m: _aligned_get(m, 'rx1day_bias', 'Rx1day_bias', 'rx1day'),              'rx1day_bias')
    _delta(lambda m: _gv(all_results[m].get('aligned'), 'psd_distance'),                   'psd_distance')
else:
    print('  [skip] dualpath and/or phase5_oracle did not run successfully')

print()
print('=' * 80)
print('SUMMARY STATUS')
print('=' * 80)
for _m in MODES_FOR_TABLE:
    _r = all_results.get(_m, {})
    print(f'  {_m:<22s} status={_r.get("status", "MISSING")}  '
          f'sigma_data={_r.get("sigma_data")}  '
          f'wallclock={_r.get("wallclock_s")}')

print()
print(f'[Cell 9] DONE. Apples-to-apples JSON => {out_path}')
